<a href="https://colab.research.google.com/github/ewilpeers/bible/blob/master/xG_Collab/01Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mēķis: atrast visas vietas kur šis minēts (takā konkordance līdzīgi)

# definīcijas

In [5]:
class GithubSecretsTikParaizsTikDrosh():
    def encode(sl, s: str) -> bytes:
        # Rotate each UTF-8 byte right by 1. Returns bytes (not str) so the
        # payload can't be mangled by text-encoding round-trips.
        result = bytearray()
        for b in s.encode('utf-8'):
            result.append(((b & 1) << 7) | (b >> 1))
        return bytes(result)

    def encode_escaped(sl, s: str) -> str:
        # Returns a Python bytes literal you can paste into source,
        # e.g. b'\x8a\xbc...'
        encoded = sl.encode(s)
        return "b'" + ''.join(f'\\x{b:02x}' for b in encoded) + "'"

    def decode(sl, s) -> str:
        # Accepts bytes (preferred) or a latin-1 str for backwards-compat.
        if isinstance(s, str):
            s = s.encode('latin-1')
        result = bytearray()
        for b in s:
            result.append(((b & 0x80) >> 7) | ((b << 1) & 0xFF))
        return result.decode('utf-8')

    def print_assignment(sl, name: str, s: str) -> None:
        # Prints a ready-to-paste line: NAME = b'\x..\x..'
        print(f"{name} = {sl.encode_escaped(s)}")


gh = GithubSecretsTikParaizsTikDrosh()

In [6]:
import pandas as pd
TOKEN_NT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\xb2\x38\x19\x39\xa0\xbc\x1c\x3a\xac\x24\xb2\x35\xaf\xa8\x9c\x18\x9c\xb3\xa1\xb8\xa2\xa2\xb2\x26\xb9\x19\x2d\x31\x3c\x39\x35\xb3\x21\xac\x26\xa1\x3d\x29\xa9\xb2\x2a\xb8\x34\x1c\x29\x32\xb8\xa1\xa4\xa1\x18\xac\xa8\xa8\x3b\x37\x26\xa6\xaa\xa4\x25\x23\xa6\x24\x9b\xa2\x33\x34\xa5\xb1\x2c\x1b'
TOKEN_NT_DATA = gh.decode(TOKEN_NT_DATA_FU_GITHUB)
TOKEN_OT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\xa0\xaa\x28\xa2\x9a\x9b\xa8\x18\xbc\x32\x9a\xa7\xb0\xb3\xb3\x33\x19\xa9\xb0\x3d\xaf\xa3\x3c\xb4\xa5\x9b\x39\x1a\x9a\x36\x3a\x99\x9a\x2b\xb4\x2c\xb6\x19\xb6\xa7\x3a\xac\xa0\xa5\xa3\x38\x33\x3b\x18\xb9\x38\xb0\x2c\x39\xbb\xb2\xb9\x99\xa1\xb3\xaa\xa7\xa6\xb4\x9a\x2d\x2c\x23\x24\xa8\xa9\xa5\xb4\x3d\x18\xb2\x25\x3d\x32\x39'
TOKEN_OT_DATA = gh.decode(TOKEN_OT_DATA_FU_GITHUB)
NT_REPO_DATA_PATH = "ewilpeers/new-testament-data"
OT_REPO_DATA_PATH = "grauziitisos/ot-west-len-data"

In [7]:

import requests
import pandas as pd
def download_csv_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	from io import StringIO
	import pandas as pd
	return pd.read_csv(StringIO(r.text), *args, **kwargs)

In [8]:
import requests
import pandas as pd
from io import BytesIO

def download_csv_from_release(repopath, tag, filename, token=None, *args, **kwargs):
    if token:
        # Private repo: go through the API with auth.
        # First, find the asset ID for this filename within the release.
        api_url = f"https://api.github.com/repos/{repopath}/releases/tags/{tag}"
        meta = requests.get(api_url, headers={
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github+json",
        })
        meta.raise_for_status()
        assets = meta.json()["assets"]
        asset = next((a for a in assets if a["name"] == filename), None)
        if asset is None:
            raise FileNotFoundError(
                f"{filename!r} not found in release {tag!r}. "
                f"Available: {[a['name'] for a in assets]}"
            )

        # Now fetch the asset bytes. octet-stream is the magic header.
        blob = requests.get(asset["url"], headers={
            "Authorization": f"token {token}",
            "Accept": "application/octet-stream",
        })
        blob.raise_for_status()
        return pd.read_csv(BytesIO(blob.content), *args, **kwargs)


    url = f"https://github.com/{repopath}/releases/download/{tag}/{filename}"
    return pd.read_csv(url, *args, **kwargs)


In [9]:
l65_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "RT65_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l24_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
hb_df = download_csv_from_release(OT_REPO_DATA_PATH, "data-v1", "biblehub_hb_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "greek-enh-lxx-apo-OT-blb.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
abp_strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "biblehub_LXX_EXT_el_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l1694_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, 'GL1694_words.csv', TOKEN_OT_DATA)


/tmp/ipykernel_82350/547696060.py:12: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(StringIO(r.text), *args, **kwargs)


In [10]:
nt_strongs_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "strongs.csv", TOKEN_NT_DATA)
nt_lv65_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "latvian_full65.csv", TOKEN_NT_DATA)
nt_l24_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_NT_DATA, dtype={'strong_num': str})
nt_l1694_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "GL1694_words.csv", TOKEN_NT_DATA)


In [11]:
hb_df.head()

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
0,1,0,וַיֹּ֨אמֶר,And said,559,"559: 1) to say, speak, utter <BR> 1a) (Qal) to...",Conj‑w|V‑Qal‑ConsecImperf‑3ms,way·yō·mer,vai·Yo·mer: And said -- Occurrence 1898 of 1948.,hosea,3
1,1,1,יְהוָ֜ה,Yahweh,3068,3068: Jehovah = the existing One<BR> 1) the pr...,N‑proper‑ms,Yah·weh,Yah·weh: Yahweh -- Occurrence 5792 of 6218.,hosea,3
2,1,2,אֵלַ֗י,to me,413,"413: 1) to, toward, unto (of motion) <BR> 2) i...",Prep|1cs,"’ê·lay,","'e·Lai,: to me -- Occurrence 415 of 446.",hosea,3
3,1,3,ע֚וֹד,again,5750,"5750: subst<BR> 1) a going round, continuance ...",Adv,‘ō·wḏ,od: again -- Occurrence 363 of 405.,hosea,3
4,1,4,לֵ֣ךְ,go,1980,"1980: 1) to go, walk, come <BR> 1a) (Qal) <BR>...",V‑Qal‑Imp‑ms,lêḵ,lech: go -- Occurrence 87 of 91.,hosea,3


In [12]:
nt_strongs_g = nt_strongs_df.groupby(["book", "chapter", "verse"])
nt_lv_g = nt_lv65_df.groupby(["book", "chapter", "verse"])
nt_l24_g = nt_l24_df.groupby(["book", "chapter", "verse"])
nt_l1694_g = nt_l1694_df.groupby(["book", "chapter", "verse"])

l65_g = l65_df.groupby(["book", "chapter", "verse"])
l24_g = l24_df.groupby(["book", "chapter", "verse"])
hb_g = hb_df.groupby(["book", "chapter", "verse"])
strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
abp_strongs_g = abp_strongs_df.groupby(["book", "chapter", "verse"])
l1694_g = l1694_df.groupby(["book", "chapter", "verse"])

In [13]:
import urllib.request, zipfile, io, os, pathlib

OT_URL = "https://github.com/ewilpeers/bible/releases/download/data-v0/bible_ot.zip"
NT_URL = "https://github.com/ewilpeers/new-testament/releases/download/data-v0/bible_nt.zip"
OT_DIR = 'bible/ot'
NT_DIR = 'bible/nt'
def fetch_and_extract(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and any(dest.iterdir()):
        return dest  # already cached
    dest.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(dest)
    return dest

ot_dir = None
if not os.path.exists(OT_DIR):
  ot_dir = fetch_and_extract(OT_URL, OT_DIR)
else:
  ot_dir =  pathlib.Path(OT_DIR)
nt_dir = None
if not os.path.exists(NT_DIR):
  nt_dir = fetch_and_extract(NT_URL, NT_DIR)
else:
  nt_dir = pathlib.Path(NT_DIR)

In [14]:
import json
records = []
leftovers = []
for book_dir in sorted(nt_dir.iterdir()):
    if not book_dir.is_dir():
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, gw in enumerate(verse.get('greek_words', [])):
                records.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'word': j, #vardi ir 0-bazeti neka panti
                    'greek': gw.get('greek', []),
                    'latvian': gw.get('latvian', []),
                })
            leftovers.append(
                {
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                }
            )

nt_df = pd.DataFrame(records)
nt_leftovers_df = pd.DataFrame(leftovers)

In [15]:
records = []
leftovers_lv = []
leftovers_gk = []
for book_dir in sorted(ot_dir.iterdir()):
    if not book_dir.is_dir():# or not book_dir.name=='1_samuel':
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()], #and f.stem=='20'],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, hw in enumerate(verse.get('hebrew_words', [])):
              #shis uzkars uz paris minutem visu lapu, bet izvadiis arii visu VD :D
                #print(hw)
                records.append({
                  'book': book,
                  'chapter': chapter,
                  'verse': i+1,
                  'word': j,
                  'hebrew': hw.get('hebrew', []),
                  'greek': hw.get('greek', []),
                  'latvian': hw.get('latvian', [])
                })

            leftovers_lv.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                })
            leftovers_gk.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_greek', [])
                })




ot_df = pd.DataFrame(records)
ot_leftovers_lv_df = pd.DataFrame(leftovers_lv)
ot_leftovers_gk_df = pd.DataFrame(leftovers_gk)

## utils

In [16]:
_LV_TABLE = str.maketrans('āčēģīķļņõŗšūžĀČĒĢĪĶĻŅÕŖŠŪŽ',
                          'acegiklnorsuzACEGIKLNORSUZ')

def strip_latvian_deep(s: str) -> str:
    return s.lower().translate(_LV_TABLE)

In [17]:
class searchExecutorSimple():
  def __init__(s, df, transFunc, case=False, regex=False):
    s.df = df
    s.case = case
    s.regex = regex
    s.transFunc = transFunc
  def search(s, searchStr):
    if s.case:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(s.transFunc(searchStr), case=False)]
    else:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(s.transFunc(searchStr), case=False)]

In [18]:
def verse_set(df, mapper = lambda x: x):
    return set(zip(df['book'], df['chapter'], df['verse']))

# darba lauks

In [19]:
nt_df.head()

,book,chapter,verse,word,greek,latvian
0,1_corinthians,1,1,0,Παῦλος,[Pāvils]
1,1_corinthians,1,1,1,κλητὸς,[aicināts]
2,1_corinthians,1,1,2,ἀπόστολος,[apustulis]
3,1_corinthians,1,1,3,Χριστοῦ,[Kristus]
4,1_corinthians,1,1,4,Ἰησοῦ,[Jēzus]


In [20]:
ot_df.head()

,book,chapter,verse,word,hebrew,greek,latvian
0,1_chronicles,1,1,0,אָדָ֥ם,[Αδαμ],[Ādams]
1,1_chronicles,1,1,1,שֵׁ֖ת,[Σηθ],[Sets]
2,1_chronicles,1,1,2,אֱנֽוֹשׁ׃,[Ενως],[Ēnošs]
3,1_chronicles,1,2,0,קֵינָ֥ן,[Καιναν],[Kenans]
4,1_chronicles,1,2,1,מַהֲלַלְאֵ֖ל,[Μαλελεηλ],[Mahalaleēls]


# 1. daļa no- vai tieša sakritība, der lieli un mazi burti, neievērot garumzīmju atšķirības

## 1.1. visvienkāršākais, esošs meklējums 1 valodas bībeles konkrētajā izdevumā

In [21]:
import unicodedata
def raccs_common(text):
    d = {
        ord('\u2019'): None,  # RIGHT SINGLE QUOTATION MARK  \u2019
        ord('\u2018'): None,  # LEFT SINGLE QUOTATION MARK   \u2018
        ord('\u201C'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('\u201D'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,
        ord(']'): None,
        ord('-'): None,
        ord("'"): None,
        ord('\u29FC'): None,  # \u29fc
        ord('\u29FD'): None,  # \u29fd
        ord('*'): None,
        ord('\u21D4'): None,  # \u21d4
        ord('\u3009'): None,  # \u3009
        ord('\u3008'): None,  # \u3008
        ord('\u203F'): None,  # \u203f
        ord('\u00AB'): None,  # \u00ab
        ord('\u00BB'): None,  # \u00bb
        ord('\u2039'): None,  # \u2039
        ord('\u203A'): None,  # \u203a
        ord('('): None,
        ord(')'): None,
        ord(';'): None,
        ord('{'): None,
        ord('}'): None,
        ord('.'): None,
        ord(','): None,
        ord('!'): None,
        ord('?'): None,
        ord(';'): None,
        ord(':'): None,
        ord('"'): None,
        ord(')'): None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
# nestrādā .... nooooooooooooooooooooooo
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        ord('᾽') : None,  # COLON
    }
    return unicodedata.normalize('NFC', text).translate(d)

def strip_greek_diacritics(s):
    return raccs_common(''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn'))

In [22]:

SEARCH1='μορφ'

strongs_df
#nt_strongs_df
abp_strongs_df

atgKaSauc = """
l65_df
l24_df
hb_df
strongs_df
abp_strongs_df
l1694_df
"""
#, case=False, regex=False
gk_ic_lxx_tr = searchExecutorSimple(strongs_df, strip_greek_diacritics, case=False, regex=False)
gk_ic_abp_tr = searchExecutorSimple(abp_strongs_df, strip_greek_diacritics, case=False, regex=False)
gk_ic_un_tr = searchExecutorSimple(nt_strongs_df, strip_greek_diacritics, case=False, regex=False)

resultats_morf_lxx_1 = gk_ic_lxx_tr.search(SEARCH1)
print(resultats_morf_lxx_1.head())

resultats_morf_abp_1 = gk_ic_abp_tr.search(SEARCH1)
print(resultats_morf_abp_1.head())

resultats_morf_un_nt_1 = gk_ic_un_tr.search(SEARCH1)
print(resultats_morf_un_nt_1.head())

          form strong_num strong_en_title    book  chapter  verse  word  \
144783   μορφὴ       3444           N-NSF  judges        8     18    23   
288811   μορφὴ       3444           N-NSF     job        4     16     8   
370916  μορφὴν       3444           N-ASF  isaiah       44     13    14   
444489   μορφή       3444           N-NSF  daniel        4     36    19   
444702   μορφὴ       3444           N-NSF  daniel        5      6     4   

       is_patched form_en strong_title  
144783        NaN     NaN          NaN  
288811        NaN     NaN          NaN  
370916        NaN     NaN          NaN  
444489        NaN     NaN          NaN  
444702        NaN     NaN          NaN  
        verse  word_idx         form             form_en strong_num  \
337417     13         8    εμόρφωσεν               forms       3445   
337434     13        25       μορφήν      the appearance       3444   
394412     16         7        μορφή  1there] appearance       3444   
438553      6      

In [23]:
len(resultats_morf_abp_1)

8

In [24]:
strip_greek_diacritics("ά'αβγδεζηθικλμνξοπρςστυφχψω")

'ααβγδεζηθικλμνξοπρςστυφχψω'

In [25]:
l1694_df.query(
    "book=='genesis' & chapter==15 & verse==6"
)

,book,chapter,verse,word_idx,form
173448,genesis,15,6,0,Un
173449,genesis,15,6,1,wiꞥſch
173450,genesis,15,6,2,tizzeja
173451,genesis,15,6,3,eekẜch
173452,genesis,15,6,4,to
173453,genesis,15,6,5,KUNGU
173454,genesis,15,6,6,un
173455,genesis,15,6,7,tas
173456,genesis,15,6,8,peelahgadija
173457,genesis,15,6,9,wiꞥꞥam


In [26]:
resultats_morf_un_nt_1.query(
    "book=='2_corinthians' & chapter==3 & verse==18"
)

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
16336,18,12,μεταμορφούμεθα,are being transformed into,3339,"3339: To transform, transfigure. From meta and...",V-PIM/P-1P,metamorphoumetha,"metamorphoumetha: To transform, transfigure. F...",2_corinthians,3


### 1.1.1. salīdzinājums

In [27]:
def printDiffCounts(set1, set2, titleString='',
                    firstName='1.', secondName='2.'):
  only_in_1 =  set1 - set2
  only_in_2 =  set2 - set1
  in_both = set1 & set2
  print(f"{titleString} skaiti: kopējie: {len(in_both)}\n {firstName}: {len(only_in_1)}\n {secondName}: {len(only_in_2)}\n")
  print(f"{titleString} SUM: {firstName}: {len(set1)} {secondName}: {len(set2)}")


In [28]:
set_lxx_morf_1 = verse_set(resultats_morf_lxx_1)
set_abp_morf_1 = verse_set(resultats_morf_abp_1)

In [29]:
printDiffCounts(set_lxx_morf_1, set_abp_morf_1,
                SEARCH1.capitalize(), 'LXX', 'ABP')

Μορφ skaiti: kopējie: 7
 LXX: 1
 ABP: 0

Μορφ SUM: LXX: 8 ABP: 7


In [30]:
SEARCH2='γαπ'
SEARCH3='στοργ'
resultats_γαπ_lxx_2 = gk_ic_lxx_tr.search(SEARCH2)
print(resultats_γαπ_lxx_2[:3])

resultats_γαπ_abp_2 = gk_ic_abp_tr.search(SEARCH2)
print(resultats_γαπ_abp_2[:3])

resultats_γαπ_un_2 = gk_ic_un_tr.search(SEARCH2)
print(resultats_γαπ_un_2[:3])

           form strong_num strong_en_title     book  chapter  verse  word  \
11353  ἀγαπητόν         27         Adj-ASM  genesis       22      2     7   
11355  ἠγάπησας         25        V-AAI-2S  genesis       22      2     9   
11618  ἀγαπητοῦ         27         Adj-GSM  genesis       22     12    29   

      is_patched form_en strong_title  
11353        NaN     NaN          NaN  
11355        NaN     NaN          NaN  
11618        NaN     NaN          NaN  
       verse  word_idx    form   form_en strong_num    strong_title      book  \
2870       5        16  αγαπάν   to love         25         to love    joshua   
8696      11         6  αγαπάν   to love         25         to love    joshua   
18810      2        15  αγάπης  the love         26  love, goodwill  jeremiah   

       chapter  
2870        22  
8696        23  
18810        2  
      verse  word      form  form_en  strong_num  \
698      14     1  ἀγαπητοί  beloved          27   
1987      1     9    ἀγάπην     lo

In [31]:
set_γαπ_lxx_2 = verse_set(resultats_γαπ_lxx_2)
set_γαπ_abp_2 = verse_set(resultats_γαπ_abp_2)
printDiffCounts(set_γαπ_lxx_2, set_γαπ_abp_2,
                SEARCH2, 'LXX', 'ABP')

γαπ skaiti: kopējie: 227
 LXX: 5
 ABP: 3

γαπ SUM: LXX: 232 ABP: 230


In [32]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==8 & verse==1"
)["form"])

'UN DEews atgahdajahs Noßs un wiẜẜo Swehru un wiẜẜo lohpu kas ar to Ꞩchꞣirſtâ bija un DEews dewe Wehju wirs Semmes un tee Uhdeni norimmahs'

at**cer**ējās -> at**gād**ājās

In [33]:
no=5
lidz=15
print(resultats_γαπ_lxx_2[["book", "chapter", "verse", "word", "strong_num", "form"]][no:lidz+1])
print(resultats_γαπ_abp_2[["book", "chapter", "verse", "word_idx", "strong_num", "form"]][no:lidz+1])


          book  chapter  verse  word  strong_num      form
14344  genesis       25     28     0          25  ἠγάπησεν
14357  genesis       25     28    13          25     ἠγάπα
17156  genesis       29     18     0          25  ἠγάπησεν
17208  genesis       29     20    16          25    ἀγαπᾶν
17373  genesis       29     30     4          25  ἠγάπησεν
17422  genesis       29     32    23          25  ἀγαπήσει
20829  genesis       34      3     9          25  ἠγάπησεν
22815  genesis       37      3     2          25     ἠγάπα
28436  genesis       44     20    29          25  ἠγάπησεν
44448   exodus       20      6     6          25  ἀγαπῶσίν
44934   exodus       21      5     6  3756|665.1   ἠγάπηκα
           book  chapter  verse  word_idx strong_num        form
22809  jeremiah       14     10         5         25    ηγάπησαν
25800  jeremiah        5     31        13         25    ηγάπησεν
29939  jeremiah       11     15         2         25   ηγαπημένη
30722  jeremiah       31      3 

In [34]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==49 & verse==18"
)["form"])

'KUNGS es gaidu us tawu Peſtiẜchanu'

In [35]:
SEARCH3='στοργ'
resultats_γαπ_lxx_3 = gk_ic_lxx_tr.search(SEARCH3)
print(resultats_γαπ_lxx_3[:3])

resultats_γαπ_abp_3 = gk_ic_abp_tr.search(SEARCH3)
print(resultats_γαπ_abp_3[:3])

resultats_γαπ_un_3 = gk_ic_un_tr.search(SEARCH3)


Empty DataFrame
Columns: [form, strong_num, strong_en_title, book, chapter, verse, word, is_patched, form_en, strong_title]
Index: []
Empty DataFrame
Columns: [verse, word_idx, form, form_en, strong_num, strong_title, book, chapter]
Index: []


In [36]:
len(resultats_γαπ_un_3)

3

In [37]:
print(resultats_γαπ_un_3[["book", "chapter", "verse", "word", "form", "form_en", "strong_num", "strong_en_title", "translit", "strong_title"]][:5])

             book  chapter  verse  word         form    form_en  strong_num  \
21168   2_timothy        3      3     0     ἄστοργοι   unloving         794   
131678     romans        1     31     2    ἀστόργους  heartless         794   
132815     romans       12     10     4  φιλόστοργοι    devoted        5387   

       strong_en_title      translit  \
21168          Adj-NMP      astorgoi   
131678         Adj-AMP     astorgous   
132815         Adj-NMP  philostorgoi   

                                             strong_title  
21168   794: Unloving, devoid of affection. Hard-heart...  
131678  794: Unloving, devoid of affection. Hard-heart...  
132815  5387: From philos and storge; fond of natural ...  


# 2. vizualizācija

**Ideja (S)**: vizualizēt (visas) vietas, atšķirīgās no kopīgajām atsevišķi (3 skati uz katru no pāriem ik meklējumā): <br><br>
piemēram, pārim 65, gliks:
<br><br>
mīl: (65, gliks, kopīgie)| (65,gliks, tikai gliks) | (65, gliks, tikai 65) <br><br>
tic: (65, gliks, kopīgie)| (65,gliks, tikai gliks) | (65, gliks, tikai 65)utt.

## 2.1. def

In [38]:
def download_txt_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	return r.text


In [39]:
MANIFEST_PATH_GREEK="mp3list_g.csv"
MANIFEST_PATH_HEBREW="mp3list_h.csv"
OT_REPO_MANIFEST_PATH = "ewilpeers/bible"
TOKEN_OT_MANF_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\x25\xba\x25\xba\xb8\xa3\xba\xbc\x9c\xb5\x3c\xa9\xaf\x38\xa0\x33\x28\xa8\x23\x18\x18\xb8\x37\x3b\xa2\x21\x19\xa9\xb1\xa7\xb5\xb0\x31\x3c\x31\x31\x3c\xac\xb0\xb4\x1a\xbb\x39\x19\xb5\x32\x1b\x18\x23\xb1\x3b\xb8\x3c\x35\x22\x27\x2b\xa3\x22\x2b\xa4\xab\xaa\x1a\x31\xb6\x1a\x31\xbb\x9a\x24\x1c'
TOKEN_OT_MANF = gh.decode(TOKEN_OT_MANF_FU_GITHUB)
if(not os.path.exists(MANIFEST_PATH_GREEK)):
  with open(MANIFEST_PATH_GREEK, 'w') as f:
    f.write(download_txt_from_private_repo(OT_REPO_MANIFEST_PATH, f"ci/{MANIFEST_PATH_GREEK}", TOKEN_OT_MANF))

if(not os.path.exists(MANIFEST_PATH_HEBREW)):
  with open(MANIFEST_PATH_HEBREW, 'w') as f:
    f.write(download_txt_from_private_repo(OT_REPO_MANIFEST_PATH, f"ci/{MANIFEST_PATH_HEBREW}", TOKEN_OT_MANF))

In [40]:
_MP3_MANIFEST_GREEK = None
_MP3_MANIFEST_HEBREW = None
AUDIO_BASE_URL_GREEK="https://t.noit.pro/strongs_p_g/"
AUDIO_BASE_URL_HEBREW="https://t.noit.pro/strongs_p/"
MANIFEST_PATH_GREEK="mp3list_g.csv"
MANIFEST_PATH_HEBREW="mp3list_h.csv"
from pathlib import Path
def load_mp3_manifest(manifest_path="mp3list_g.csv", language="greek"):
    """Load mp3 manifest from CSV. Falls back to empty set if file missing."""
    global _MP3_MANIFEST_GREEK, _MP3_MANIFEST_HEBREW
    manifest = set()
    if not (_MP3_MANIFEST_GREEK if language == "greek" else _MP3_MANIFEST_HEBREW):
        p = Path(manifest_path)
        if p.exists():
            df = pd.read_csv(p)
            manifest = set(df["filename"].str.strip())
            print(f"  📋 Loaded {len(manifest)} entries from {p}")
        else:
            print(f"  ⚠️ {p} not found — {language} audio players will be disabled")
            manifest = set()
    if language == "greek":
        _MP3_MANIFEST_GREEK = manifest
    else:
        _MP3_MANIFEST_HEBREW = manifest


def make_audio_players(strong_num_raw,
                       v_num,
                       word_idx,
                       prependix="h",
                       subword_idx='',
                       br_prepend_start=True,
                       manifest_path=None):
    """
    Build inline audio play buttons resolved via manifest lookup.

    prependix: "h" for Hebrew (default), "g" for Greek
    manifest_path: override CSV path; defaults per language
    """
    global _MP3_MANIFEST_GREEK, _MP3_MANIFEST_HEBREW

    if prependix == "g":
        if _MP3_MANIFEST_GREEK is None:
            load_mp3_manifest(manifest_path or MANIFEST_PATH_GREEK, language="greek")
        manifest = _MP3_MANIFEST_GREEK
        base_url = AUDIO_BASE_URL_GREEK
    else:
        if _MP3_MANIFEST_HEBREW is None:
            load_mp3_manifest(manifest_path or MANIFEST_PATH_HEBREW, language="hebrew")
        manifest = _MP3_MANIFEST_HEBREW
        base_url = AUDIO_BASE_URL_HEBREW

    if not strong_num_raw:
        return ""
    try:
        sn = int(strong_num_raw)
    except (ValueError, TypeError):
        return ""
    if sn <= 0:
        return ""

    skey = f"{prependix}{sn:04d}"

    variants = []
    if f"{skey}.mp3" in manifest:
        variants.append((f"{base_url}/{skey}.mp3", ""))
    vi = 2
    while f"{skey}-{vi}.mp3" in manifest:
        variants.append((f"{base_url}/{skey}-{vi}.mp3", f" {vi}"))
        vi += 1

    if not variants:
        return ""

    out = "<br>" if br_prepend_start else ""
    for src, label in variants:
        play_label = f"▶{label}"
        out += (
            f'<button data-label="{play_label}" data-suffix="{label}" '
            f'style="font-size:0.8em;padding:1px 5px;cursor:pointer" '
            f"onclick=\"playStrong('{src}', this)\">"
            f'{play_label}</button> '
        )
    return out
def getHead():
      css = """
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; line-height: 1.6; color: #333; max-width: 1600px; margin: 0 auto; padding: 20px; background-color: #f8f9fa; }
        h1 { text-align: center; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        .verse-container { background-color: white; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-bottom: 40px; padding: 25px; border-left: 5px solid #3498db; }
        .verse-header { font-weight: bold; color: #2c3e50; background-color: #ecf0f1; padding: 8px 15px; border-radius: 20px; margin-bottom: 15px; }
        .verse-lines { display: flex; flex-wrap: wrap; gap: 15px; margin-bottom: 20px; }
        .line-box { flex: 1 1 45%; min-width: 300px; }
        .line-label { font-weight: bold; color: #16a085; margin-bottom: 5px; }
        .line-content { background-color: #f0f7f4; padding: 12px; border-radius: 8px; border: 1px solid #bdc3c7; font-size: 1.1em; }
        .hebrew-line { background-color: #f0f0f0; font-family: 'Times New Roman', serif; }

        /* Table Styles */
        .mapping-table { width: 100%; border-collapse: collapse; margin-top: 15px; font-size: 0.9em; }
        .mapping-table th { background-color: #3498db; color: white; padding: 12px; text-align: left; position: sticky; top: 0; }
        .mapping-table td { padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; }

        .hebrew-word { font-weight: bold; color: #8e44ad; font-size: 1.1em; }
        .hebrew-form { color: #7f8c8d; font-weight: normal; font-size: 0.85em; }
        .latvian-word { font-weight: bold; color: #27ae60; }

        /* Morphology Tooltip Style */
        .morph-info {
            font-style: italic;
            color: #e67e22;
            cursor: text;
            border-bottom: 1px dotted #e67e22;
            display: inline-block;
        }
        .definition-cell { color: #555; font-size: 0.85em; line-height: 1.3; max-width: 400px; }
        .index-badge { display: inline-block; width: 25px; height: 25px; background-color: #e74c3c; color: white; border-radius: 50%; text-align: center; line-height: 25px; margin-right: 8px; }
        /* group greek play button(s) together with the word */
        .greek-audio-group {
          display: inline-flex;
          align-items: baseline;
          white-space: nowrap;
        }
/* ⓘ disclosure widget  */
    .verse-info {
        display: inline;
        position: relative;
    }
    .verse-info summary {
        display: inline-flex;
        align-items: center;
        justify-content: center;
        width: 22px;
        height: 22px;
        border-radius: 50%;
        background: #3498db;
        color: white;
        font-size: 13px;
        font-weight: bold;
        cursor: pointer;
        list-style: none;           /* hide default triangle — Firefox */
        margin-left: 8px;
        vertical-align: middle;
        user-select: none;
        transition: background 0.2s;
        line-height: 1;
    }
    .verse-info summary::-webkit-details-marker { display: none; }  /* Chrome/Safari */
    .verse-info summary::marker { content: ''; font-size: 0; }     /* Firefox fallback */
    .verse-info summary:hover,
    .verse-info summary:focus-visible {
        background: #2980b9;
        outline: 2px solid #2980b9;
        outline-offset: 2px;
    }
    .verse-info[open] summary { background: #2c3e50; }

    .verse-info .info-popup {
        display: block;
        margin-top: 10px;
        padding: 10px 14px;
        background: #eaf4fb;
        border: 1px solid #aed6f1;
        border-radius: 8px;
        font-size: 0.95em;
        color: #2c3e50;
        line-height: 1.5;
    }
    .info-popup .info-label {
        font-weight: bold;
        color: #2471a3;
        margin-bottom: 2px;
    }
 .diffV { background-color: #F6AA11; }

        /* gothic old print render */
@font-face {
    font-family: 'UnifrakturMaguntia';
    src: url('../fonts/unifrakturmaguntia-webfont.woff2') format('woff2'),
         url('../fonts/unifrakturmaguntia-webfont.woff') format('woff'),
         url('../fonts/unifrakturmaguntia-webfont.ttf') format('truetype');
    font-weight: normal;
    font-style: normal;
    font-display: swap;
}
.frankfurt-line {
    font-family: 'UnifrakturMaguntia', 'Times New Roman', serif;
   /* background-color: #fdf6ec; */
}
    </style>
    """
      javascript="""
<script type="text/javascript">
var _activeBtn = null;

function playStrong(src, btn) {
  var player = document.getElementById('shared-player');

  // if clicking the same button while playing — stop it
  if (_activeBtn === btn && !player.paused) {
    player.pause();
    player.currentTime = 0;
    btn.textContent = btn.dataset.label;
    _activeBtn = null;
    return;
  }

  // reset previous button if any
  if (_activeBtn) {
    _activeBtn.textContent = _activeBtn.dataset.label;
  }

  _activeBtn = btn;
  btn.textContent = '⏹' + (btn.dataset.suffix || '');
  player.src = src;
  player.play();
}

document.addEventListener('DOMContentLoaded', function() {
  document.getElementById('shared-player').addEventListener('ended', function() {
    if (_activeBtn) {
      _activeBtn.textContent = _activeBtn.dataset.label;
      _activeBtn = null;
    }
  });
});
</script>
    """
      return f"<!DOCTYPE html><html><head><meta charset='UTF-8'>{css}{javascript}</head><body>"

def chapter_to_html_render(data, html):
    if not data or len(data) < 1:
        return ""

    audio_tag = '<audio id="shared-player"></audio>'
    book_title = data[0].get('book', 'Bible').capitalize()
    chapter = data[0].get('chapter', '')
    html += f"<h1>📖 {book_title} Chapter {chapter}</h1>"
    html += audio_tag

    for verse_data in data:
        v_num = verse_data.get('verse')
        locus = f"{book_title} {chapter}:{v_num}"

        html += f'<div class="verse-container">'
        html += f'<div class="verse-header"><span class="index-badge">{v_num}</span> {locus}</div>'

        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇮🇱 Hebrew:</div>
                <div class="line-content hebrew-line">{verse_data.get('hebrew_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1694):</div>
                <div class="line-content frankfurt-line">{verse_data.get('latvian_text_full_original_1694', '')}</div>
            </div>
        </div>
        '''
        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇬🇷 Greek (LXX augumented with ABP):</div>
                <div class="line-content hebrew-line">{verse_data.get('greek_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1965):</div>
                <div class="line-content">{verse_data.get('latvian_text_full_original_65', '')}
                <details class="verse-info">
                    <summary aria-label="Show 2024 Latvian rendering" title="2024 rendering">4</summary>
                    <div class="info-popup">
                        <div class="info-label">🇱🇻 Latvian (2024):</div>
                        {verse_data.get('latvian_text_full_original_24', '')}
                    </div>
                </details>
                </div>
            </div>
        </div>
        '''
    return html

def getTail():
    html += "</body></html>"
    return html
def render_verse(verse_text, needle, transformFun=lambda x: x):
    t_text = transformFun(verse_text)
    t_needle = transformFun(needle)
    result = []
    i = 0
    while i < len(verse_text):
        if t_text[i:i+len(t_needle)] == t_needle:
            result.append("<span class=\"diffV\">" + verse_text[i:i+len(t_needle)] + "</span>")
            i += len(t_needle)
        else:
            result.append(verse_text[i])
            i += 1
    return ''.join(result)

In [41]:
def renderResultsToHTML(results):

    html = getHead()
    for verse_data in results:
        html += chapter_to_html_render(verse_data, html)
    html += getTail()
    return html



## 2.2. action by Claude

In [42]:
# ====== fixes to helpers from 2.1 ======

def getTail():
    return "</body></html>"

def chapter_to_html_render(data):
    """Render a list of verse-dicts belonging to a single book+chapter."""
    if not data:
        return ""
    html = ""
    audio_tag = '<audio id="shared-player"></audio>'
    book_title = data[0].get('book', 'Bible').replace('_', ' ').capitalize()
    chapter = data[0].get('chapter', '')
    html += f"<h1>📖 {book_title} Chapter {chapter}</h1>"
    html += audio_tag

    for verse_data in data:
        v_num = verse_data.get('verse')
        locus = f"{book_title} {chapter}:{v_num}"

        html += '<div class="verse-container">'
        html += f'<div class="verse-header"><span class="index-badge">{v_num}</span> {locus}</div>'

        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇮🇱 Hebrew:</div>
                <div class="line-content hebrew-line">{verse_data.get('hebrew_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1694):</div>
                <div class="line-content frankfurt-line">{verse_data.get('latvian_text_full_original_1694', '')}</div>
            </div>
        </div>
        '''
        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇬🇷 Greek (LXX + ABP):</div>
                <div class="line-content hebrew-line">{verse_data.get('lxx_text_full_original', '')}</div>
                <div class="line-content hebrew-line">{verse_data.get('abp_text_full_original', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian (1965):</div>
                <div class="line-content">{verse_data.get('latvian_text_full_original_65', '')}
                <div class="verse-info">
                    <div class="info-popup">
                        <div class="info-label">🇱🇻 Latvian (2024):</div>
                        {verse_data.get('latvian_text_full_original_24', '')}
                    </div>
                </div>
                </div>
            </div>
        </div>
        '''
        html += '</div>'
    return html

In [43]:
# ====== 2.2. action — build verse-dicts from the dataframes ======

from functools import lru_cache

# canonical book ordering from ot_df (falls back to alpha if not present)
try:
    _BOOK_ORDER = {b: i for i, b in enumerate(ot_df['book'].drop_duplicates())}
except NameError:
    _BOOK_ORDER = {}

def _sort_key(bcv):
    book, ch, v = bcv
    return (_BOOK_ORDER.get(book, 10_000), book, ch, v)


def _verse_text(df, book, ch, v):
    """Space-joined form column for a single verse, or '' if missing."""
    sub = df[(df['book'] == book) & (df['chapter'] == ch) & (df['verse'] == v)]
    if sub.empty:
        return ''
    return ' '.join(sub['form'].astype(str).tolist())


def _hb_text(book, ch, v):
    sub = hb_df[(hb_df['book'] == book) & (hb_df['chapter'] == ch) & (hb_df['verse'] == v)]
    if sub.empty:
        return ''
    # hb_df uses 'hebrew' or 'form'? — pick whichever is there
    col = 'hebrew' if 'hebrew' in sub.columns else ('form' if 'form' in sub.columns else None)
    return ' '.join(sub[col].astype(str).tolist()) if col else ''

def _lxx_text(book, ch, v):
    sub = strongs_df[(strongs_df['book'] == book) & (strongs_df['chapter'] == ch) & (strongs_df['verse'] == v)]
    if not sub.empty:
        return ' '.join(sub['form'].astype(str).tolist())
    return ''

def _abp_text(book, ch, v):
    sub = abp_strongs_df[(abp_strongs_df['book'] == book) & (abp_strongs_df['chapter'] == ch) & (abp_strongs_df['verse'] == v)]
    if not sub.empty:
          return ' '.join(sub['form'].astype(str).tolist())
    return ''


def build_verse_dicts(bcv_set, needle,
                      needle_1694=None, needle_24=None,
                      needle_lxx = None, needle_abp = None,
                      transform_1694= lambda x:x, transform_24=lambda x:x,
                      transform_65 = lambda x: x,
                      transform_lxx = lambda x: x,
                      transform_abp = lambda x: x,
                      highlight_accent_insensitive=True):
    """
    bcv_set: iterable of (book, chapter, verse)
    needle:  the 1965 / 2024 search term (e.g. 'tic' or 'mīl')
    needle_1694: variant for Glik (e.g. 'tiz' or 'mihl'); defaults to needle
    Returns a list of verse-dicts in canonical order, ready for chapter_to_html_render.
    """
    needle_1694 = needle_1694 if needle_1694 is not None else needle
    needle_24 = needle_24 if needle_24 is not None else needle

    # accent-insensitive (deep-strip) for 1965/2024; case-sensitive lambda for Glik
    #tr_modern = strip_latvian_deep if highlight_accent_insensitive else (lambda x: x)
    #tr_glik   = (lambda x: x.lower())  # Glik data uses mixed case; normalise to lower for matching

    rows = []
    for book, ch, v in sorted(bcv_set, key=_sort_key):
        t65   = _verse_text(l65_df,   book, ch, v)
        t24   = _verse_text(l24_df,   book, ch, v)
        t1694 = _verse_text(l1694_df, book, ch, v)
        thb   = _hb_text(book, ch, v)
        tlxx   = _lxx_text(book, ch, v)
        tabp = _abp_text(book, ch, v)

        rows.append({
            'book': book,
            'chapter': ch,
            'verse': v,
            'hebrew_text_full_original':        thb,
            'lxx_text_full_original':         render_verse(tlxx, needle_lxx, transform_lxx),
            'abp_text_full_original':         render_verse(tabp, needle_abp, transform_abp),
            'latvian_text_full_original_65':    render_verse(t65,   needle,      transform_65),
            'latvian_text_full_original_24':    render_verse(t24,   needle_24,   transform_24),
            'latvian_text_full_original_1694':  render_verse(t1694, needle_1694, transform_1694),
        })
    return rows


def group_by_chapter(verse_dicts):
    """Chunk a flat list of verse-dicts into per-(book,chapter) lists, preserving order."""
    out = []
    cur_key = None
    cur_list = []
    for d in verse_dicts:
        k = (d['book'], d['chapter'])
        if k != cur_key:
            if cur_list:
                out.append(cur_list)
            cur_list = []
            cur_key = k
        cur_list.append(d)
    if cur_list:
        out.append(cur_list)
    return out

In [44]:
# replacement for the broken renderResultsToHTML in cell 58

def renderResultsToHTML(verse_dicts, heading=None):
    html = getHead()
    if heading:
        html += f"<h1 style='text-align:center'>{heading}</h1>"
    for chap in group_by_chapter(verse_dicts):
        html += chapter_to_html_render(chap)
    html += getTail()
    return html


def render_comparison_triplet(set_a, set_b, needle,
                              name_a='A', name_b='B',
                              needle_1694=None, needle_24=None,
                              transform_1694= lambda x:x, transform_24=lambda x:x,
                              transform_65 = lambda x: x,
                              transform_lxx = lambda x: x,
                              transform_abp = lambda x: x,
                              highlight_accent_insensitive=True,
                              fileNamePrefix=""):
    """
    Build the three HTML views described in 2. vizualizācija:
      (kopīgie, tikai A, tikai B)
    """
    common    = set_a & set_b
    only_in_a = set_a - set_b
    only_in_b = set_b - set_a

    build = lambda s: build_verse_dicts(
        s, needle,
        needle_1694=needle_1694, needle_24=needle_24,
        highlight_accent_insensitive=highlight_accent_insensitive,
        transform_1694=transform_1694, transform_24=transform_24,
        transform_65=transform_65, transform_lxx=transform_lxx,
        transform_abp = transform_abp
    )
    return {
        f'{fileNamePrefix} tilts {name_a} UN  {name_b} ({needle}) abos':       renderResultsToHTML(build(common),    heading=f'{needle!r}: kopīgie ({name_a} ∩ {name_b}) — {len(common)}'),
        f'{fileNamePrefix} tikai {name_a} NE {name_b} ({needle})':               renderResultsToHTML(build(only_in_a), heading=f'{needle!r}: tikai {name_a} NE {name_b} — {len(only_in_a)}'),
        f'{fileNamePrefix} tikai {name_b} NE {name_a} ({needle})':               renderResultsToHTML(build(only_in_b), heading=f'{needle!r}: tikai {name_b} NE {name_a} — {len(only_in_b)}'),
    }

In [45]:
import pathlib
def write_rendered(view):
  out_dir = pathlib.Path('render_out'); out_dir.mkdir(exist_ok=True)
  for title, html in view.items():
      safe = title.replace(' ', '_').replace('/', '_').replace('∩', 'and').replace("'", '')
      (out_dir / f'{safe}.html').write_text(html, encoding='utf-8')
      print(f'wrote {safe}.html  ({len(html):,} chars)')

# 3. call Claude 's render (me)

In [ ]:
from tqdm.notebook import tqdm
set_γαπ_lxx_2 = verse_set(resultats_γαπ_lxx_2)
set_γαπ_abp_2 = verse_set(resultats_γαπ_abp_2)
for v1, v2, v3 in tqdm(zip([('LXX', 'ABP'),
                       #('1965', '1694'),
                      # ('2024', '1694')
                      ],
 [(set_γαπ_lxx_2, set_γαπ_abp_2),
  #(set_65_mil_4t, set_1694_mil_4t),
  #(set_24_mil_4t, set_1694_mil_4t),
  ],
  [(SEARCH2, SEARCH2),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih')),
   #(SEARCH3.replace('ļ', 'l'), SEARCH3.replace('ī', 'ih'))
   ])):

  vview = render_comparison_triplet(
      v2[0], v2[1],
      needle=v3[0],                              # 'mīl'
      needle_1694=v3[1],      # 'mihl'
      name_a=v1[0], name_b=v1[1],
      transform_lxx=lambda x: '' if x==None else strip_greek_diacritics(x).lower(),
      transform_abp=lambda x:  '' if x==None else strip_greek_diacritics(x).lower(),
      highlight_accent_insensitive=False,          # 4s = case/accent sensitive
  )
  write_rendered(vview)



0it [00:00, ?it/s]